**FASE 7: EL EXPERTO CINEMÁTICO (Telemetría Inercial y TinyML)**

Autor: Andoni Cabrera Fernández

Descripción: En este cuaderno se desarrolla el segundo pilar de la arquitectura multimodal del TFG. Ante los posibles puntos ciegos de la visión artificial, se diseña un "Experto Cinemático" basado en la telemetría del vehículo (Acelerómetro y Giroscopio).
Se procesa el *Driving Behavior Dataset*, aplicando ingeniería de características (*Feature Engineering*) mediante Ventanas Temporales Móviles de 3 segundos y el cálculo de la Magnitud Vectorial (para lograr invarianza espacial del sensor). El modelo predictivo se basa en Bosques Aleatorios (*Random Forest*), optimizado para su despliegue en hardware de recursos limitados.

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Conectar con Google Drive
drive.mount('/content/drive')
print("\nLibrerías importadas y entorno configurado correctamente.")

**1. CARGA DEL DATASET Y PRE-PROCESAMIENTO BASE**

In [ ]:
# Rutas (Ajusta la carpeta según tu Drive)
RUTA_TRAIN = '/content/drive/MyDrive/Telemetria/train_motion_data.csv'
RUTA_TEST = '/content/drive/MyDrive/Telemetria/test_motion_data.csv'

# Carga y unificación de los CSV para partición propia
df_train = pd.read_csv(RUTA_TRAIN)
df_test = pd.read_csv(RUTA_TEST)
df_all = pd.concat([df_train, df_test], ignore_index=True)
df_all.columns = df_all.columns.str.strip()

features = ['AccX', 'AccY', 'AccZ', 'GyroX', 'GyroY', 'GyroZ']
target = 'Class'

# Mapeo a valores numéricos (0: Normal, 1: Slow, 2: Aggressive)
# La clase "Aggressive" simula los espasmos o volantazos característicos del micro-sueño
mapeo_clases = {'NORMAL': 0, 'SLOW': 1, 'AGGRESSIVE': 2}
df_all['Label'] = df_all[target].map(mapeo_clases)
df_all = df_all.dropna(subset=features + ['Label'])

print(f"Dataset cargado con éxito. Total de muestras crudas: {len(df_all)}")
display(df_all.head())

**2. FEATURE ENGINEERING (Magnitud Vectorial y Rolling Windows)**

In [ ]:
# 1. Invarianza Espacial: Magnitud Vectorial (Fuerza total sin importar la orientación del sensor)
df_all['Acc_Mag'] = np.sqrt(df_all['AccX']**2 + df_all['AccY']**2 + df_all['AccZ']**2)
df_all['Gyro_Mag'] = np.sqrt(df_all['GyroX']**2 + df_all['GyroY']**2 + df_all['GyroZ']**2)

features_ampliadas = features + ['Acc_Mag', 'Gyro_Mag']

def crear_ventana_temporal(df, window_size=6):
    """ window_size=6 equivale a 3 segundos (muestreo a 2Hz) """
    dfs = []
    for c in df['Class'].unique():
        sub_df = df[df['Class'] == c].copy()

        # Extracción estadística
        for f in features_ampliadas:
            sub_df[f'{f}_mean'] = sub_df[f].rolling(window_size).mean()
            sub_df[f'{f}_std']  = sub_df[f].rolling(window_size).std()
            sub_df[f'{f}_max']  = sub_df[f].rolling(window_size).max()
            sub_df[f'{f}_min']  = sub_df[f].rolling(window_size).min()

        dfs.append(sub_df.dropna())
    return pd.concat(dfs, ignore_index=True)

print("Aplicando Ventanas Temporales (Filtro de 3 segundos)...")
df_win = crear_ventana_temporal(df_all, window_size=6)

nuevas_features = [c for c in df_win.columns if any(f in c for f in features_ampliadas) and '_' in c]
X = df_win[nuevas_features].values
y = df_win['Label'].values

print(f"Transformación completada. Dataset final: {len(X)} muestras con {X.shape[1]} características.")

**3. ENTRENAMIENTO Y EVALUACIÓN (TINYML)**

In [ ]:
# División estricta (Hold-out Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Random Forest acotado para sistemas integrados
modelo_rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
modelo_rf.fit(X_train, y_train)

# Predicción y Evaluación
y_pred = modelo_rf.predict(X_test)
precision = accuracy_score(y_test, y_pred)

print(f"=== EXACTITUD GLOBAL (ACCURACY): {precision * 100:.2f}% ===\n")
print(classification_report(y_test, y_pred, target_names=['NORMAL (0)', 'SLOW (1)', 'AGGRESSIVE (2)']))

# Visualización
plt.figure(figsize=(7, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['NORMAL', 'SLOW', 'AGGRESSIVE'],
            yticklabels=['NORMAL', 'SLOW', 'AGGRESSIVE'])
plt.title('Matriz de Confusión - Experto Cinemático')
plt.ylabel('Estado Real'); plt.xlabel('Estado Predicho')
plt.show()

# Exportar modelo
joblib.dump(modelo_rf, '/content/drive/MyDrive/Telemetria/experto_cinematico.pkl')